# NB 1.1 &mdash; Del perceptró al perceptró multicapa

**MP 5149** &mdash; Desenvolupament de components software per a sistemes d'aprenentatge automàtic

**UT5** &mdash; Introducció a les xarxes neuronals supervisades

---
### Què farem avui

Al mòdul 5134 heu entrenat el primer model amb scikit-learn: una regressió que
aprenia uns coeficients a partir de les mesures dels pingüins. Allà vau veure que
qualsevol model de scikit-learn es fa servir igual: s'entrena amb `fit`, fa
prediccions amb `predict` i s'avalua amb `score`.

Aquí farem servir **la mateixa biblioteca i les mateixes tres ordres** per a una família
d'algorismes diferent: les **xarxes neuronals**. No les programarem per dins ni
farem matemàtiques: les **entrenarem amb scikit-learn** i, per entendre què fan,
**les mirarem**: dibuixarem les fronteres que aprenen, les corbes d'entrenament i
el que passa quan toquem els seus hiperparàmetres.

El fil del notebook segueix la història, perquè cada pas va néixer per resoldre
el problema que el pas anterior no sabia resoldre:

| Any | Idea | Què resol | Què no resol |
|---|---|---|---|
| 1943 | Neurona de McCulloch i Pitts | Calcula funcions lògiques | No aprèn: els valors els poses tu |
| 1958 | Perceptró de Rosenblatt | Aprèn els pesos a partir d'exemples | Només separa amb una recta |
| 1969 | El problema de la XOR | &mdash; | Frena la recerca durant anys |
| 1986 | Perceptró multicapa i backpropagation | Aprèn problemes no lineals | &mdash; |

### Preguntes que has de saber respondre en acabar

1. Què és una neurona artificial i quines peces té?
2. Com aprèn un perceptró, i per què no pot aprendre la XOR?
3. Què aporta una capa oculta que un model lineal no té?
4. Què és el gradient descent i què passa si el learning rate és massa gran?
5. Què fa la backpropagation, explicat sense fórmules?
6. **En què es diferencien les xarxes neuronals dels algorismes clàssics que veieu al 5134?**

## 0. Preparació

Tot el notebook funciona amb quatre biblioteques que ja coneixeu del 5134:
**NumPy** per a les operacions amb taules de números, **pandas** per carregar les
dades, **Matplotlib** per dibuixar i **scikit-learn**, que és la protagonista: hi
trobarem el perceptró (`Perceptron`) i el perceptró multicapa (`MLPClassifier` i
`MLPRegressor`), amb la interfície `fit`, `predict` i `score` de sempre.

Les dades dels pingüins les carreguem del repositori del mòdul 5134, per URL, com
sempre. No cal descarregar res.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Copy of the palmerpenguins data (Gorman et al., 2014) kept in the MP 5134 repository
URL_DATA = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/disseny-avaluacio-models-ml/refs/heads/main/UT01-Entorn_de_treball_primer_model/penguins/penguins.csv"

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

La cel·la següent defineix una funció auxiliar per **dibuixar la frontera de
decisió** d'un classificador: pinta de cada color la zona del pla que el model
assignaria a cada classe i hi posa els punts a sobre.

No cal que l'entenguis ara. Funciona així: crea una graella de milers de punts
que cobreix el gràfic, demana al model la predicció de cadascun i acoloreix el
resultat. Rep una **funció** de predicció, no un model, i així la podrem fer
servir tant amb els models de scikit-learn (`model.predict`) com amb neurones fetes
a mà.

In [ ]:
def plot_decision_boundary(predict, X, y, ax=None, title=None, margin=0.5, resolution=300):
    """Colour each region of the plane with the class that `predict` assigns to it.

    predict : callable that receives an (n, 2) array and returns 0/1 labels.
    X, y    : samples to draw on top of the regions.
    """
    ax = ax or plt.gca()
    X = np.asarray(X, dtype=float)
    x_min, x_max = X[:, 0].min() - margin, X[:, 0].max() + margin
    y_min, y_max = X[:, 1].min() - margin, X[:, 1].max() + margin
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution),
                         np.linspace(y_min, y_max, resolution))
    grid = np.column_stack([xx.ravel(), yy.ravel()])
    zz = np.asarray(predict(grid)).reshape(xx.shape)

    ax.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5], colors=["#9ecae1", "#fdae6b"], alpha=0.5)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=45, zorder=3)
    ax.grid(False)
    if title:
        ax.set_title(title)
    return ax

## 1. Dues famílies d'algorismes

Abans de començar, una idea que us ha d'acompanyar tot el curs, perquè feu els
dos mòduls alhora.

Al **5134** estudiareu els **algorismes clàssics d'aprenentatge automàtic**:
regressió lineal i logística, k-nearest neighbors, decision trees, random forest, màquines de
suport vectorial, k-means. Cadascun és **una idea concreta** sobre com separar o
ajustar les dades: una recta, una sèrie de preguntes, el marge més ample, els
veïns més propers. Funcionen molt bé amb taules de dades, s'entrenen de pressa i
molts es poden llegir i explicar.

Al **5149** estudiarem les **xarxes neuronals**, que són una família a part.
Una xarxa no és una idea sobre la forma de la frontera. Totes es construeixen
igual: **moltes neurones molt simples, totes iguals, connectades en capes**. Una
neurona sola fa poca cosa; la potència ve de **com les connectes** i del fet que
**totes s'entrenen amb la mateixa recepta**.

Hi ha una diferència que farà de fil conductor d'aquest notebook i que convé
que tinguis present des d'ara:

> **Als algorismes clàssics, les característiques les prepares tu. A les xarxes
> neuronals, les capes ocultes les aprenen.**

Ara sona abstracte. A la secció 5 ho veurem amb un exemple de quatre punts, i a
la secció 6 veurem la xarxa fent-ho sola.

## 2. La inspiració: la neurona biològica

El cervell humà té uns 86.000 milions de neurones, i cadascuna es connecta amb
milers d'altres. Una neurona, simplificant molt, té tres parts:

- Les **dendrites** reben senyals d'altres neurones.
- El **cos cel·lular** (soma) *suma* aquests senyals.
- Si la suma supera un **llindar**, la neurona **es dispara** i envia un impuls
  per l'**axó** cap a les neurones següents. Si no el supera, no fa res. És tot
  o res.

Les connexions entre neurones, les **sinapsis**, no són totes iguals: n'hi ha de
fortes i de febles, n'hi ha que exciten i n'hi ha que inhibeixen. I sabem, des de
Donald Hebb (1949), que **aprendre té a veure amb reforçar o afeblir aquestes
connexions**.

D'aquí surten les tres idees que copiarà la neurona artificial:

| Biologia | Neurona artificial |
|---|---|
| Senyals que arriben per les dendrites | **Entrades** $x_1, x_2, \dots$ |
| Força de cada sinapsi | **Pesos** $w_1, w_2, \dots$ |
| Llindar de disparament | **Biaix** i **funció d'activació** |
| Aprendre = canviar la força de les sinapsis | **Entrenar = ajustar els pesos** |

Una advertència honesta: les xarxes neuronals artificials **s'inspiren** en el
cervell, però no el simulen. La neurona biològica és molt més complexa. La
metàfora serveix per arrencar, i prou.

## 3. 1943: la neurona de McCulloch i Pitts

El neuròleg Warren McCulloch i el lògic Walter Pitts van proposar el primer model
matemàtic d'una neurona. És extremadament simple:

- Les entrades valen **0 o 1** (arriba senyal o no).
- La neurona compta quantes entrades excitadores estan actives.
- Si el recompte arriba al **llindar**, la sortida és 1. Si no, és 0.
- Si arriba qualsevol entrada **inhibidora**, la sortida és 0 passi el que passi.

La seva gran troballa va ser que, **combinant neurones com aquestes, es pot
calcular qualsevol funció lògica**. El cervell, per primera vegada, es descrivia
com una màquina de càlcul.

In [ ]:
def mcculloch_pitts(excitatory, threshold, inhibitory=()):
    """McCulloch-Pitts neuron (1943): binary inputs, binary output, fixed threshold."""
    if any(inhibitory):
        return 0
    return int(sum(excitatory) >= threshold)


TRUTH_TABLE = [(0, 0), (0, 1), (1, 0), (1, 1)]

print("x1 x2 | AND  OR")
print("------+--------")
for x1, x2 in TRUTH_TABLE:
    and_out = mcculloch_pitts([x1, x2], threshold=2)
    or_out = mcculloch_pitts([x1, x2], threshold=1)
    print(f" {x1}  {x2} |  {and_out}    {or_out}")

La mateixa neurona fa dues coses diferents segons el llindar:

- Amb **llindar 2**, necessita que les dues entrades estiguin actives: és una
  porta **AND**.
- Amb **llindar 1**, en té prou amb una: és una porta **OR**.

Amb una entrada inhibidora també podem fer la negació:

In [ ]:
print("x | NOT")
print("--+----")
for x in (0, 1):
    # No excitatory inputs, threshold 0: fires by default unless x inhibits it
    print(f"{x} |  {mcculloch_pitts([], threshold=0, inhibitory=[x])}")

Amb AND, OR i NOT es pot construir qualsevol circuit lògic, i per tant qualsevol
càlcul. Impressionant per al 1943.

Però fixa't en una cosa: **el llindar l'hem triat nosaltres**. Hem pensat quina
funció volíem i hem posat el número que la feia funcionar. Aquesta neurona **no
aprèn res**. És programació clàssica amb forma de neurona:

`DADES + REGLES → RESPOSTES`

A la UT1 del 5134 vau veure que l'aprenentatge automàtic gira aquest esquema:
`DADES + RESPOSTES → REGLES`. Això és el que farà el perceptró quinze anys després.

## 4. 1958: el perceptró de Rosenblatt

Frank Rosenblatt, psicòleg del Cornell Aeronautical Laboratory, va fer tres canvis
a la neurona de McCulloch i Pitts:

1. Les entrades poden ser **qualsevol número**, no només 0 o 1.
2. Cada entrada té un **pes** que diu com d'important és, i que pot ser positiu o negatiu.
3. I el més important: **els pesos s'aprenen a partir d'exemples**.

No només en va escriure la teoria. En va construir una màquina, el **Mark I
Perceptron**, amb una «retina» de 400 cèl·lules fotoelèctriques i els pesos fets
amb potenciòmetres que uns motors elèctrics anaven girant mentre la màquina
aprenia a reconèixer formes.

### 4.1 Com calcula

El perceptró fa dues operacions:

1. Una **suma ponderada** de les entrades, més el biaix:
   $\quad z = w_1 x_1 + w_2 x_2 + b$
2. Una **funció step**: si $z \geq 0$ la sortida és 1, i si no, 0.

El **biaix** $b$ fa el paper del llindar: com més negatiu, més li costa a la
neurona disparar-se.

Posem-hi números. Si triem $w_1 = 1$, $w_2 = 1$ i $b = -1{,}5$:

In [ ]:
X_logic = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)

weights = np.array([1.0, 1.0])
bias = -1.5

z = X_logic @ weights + bias          # weighted sum for the 4 rows at once
output = np.where(z >= 0, 1, 0)       # step function

for (x1, x2), zi, out in zip(X_logic, z, output):
    print(f"x = ({x1:.0f}, {x2:.0f})   z = {x1:.0f}·1 + {x2:.0f}·1 - 1.5 = {zi:+.1f}   ->  output {out}")

Torna a ser una porta AND, però ara amb pesos i biaix en lloc d'un recompte.

Dues coses de codi que convé fixar, perquè les farem servir a tot el notebook:

**`X_logic @ weights`** és el producte de matrius de NumPy. `X_logic` té 4 files
i 2 columnes, `weights` té 2 valors, i el resultat són 4 sumes ponderades, una
per fila. **Calculem totes les mostres de cop**, sense cap bucle. En direm
*vectoritzar*, i és el que fa que les xarxes siguin ràpides.

**`np.where(z >= 0, 1, 0)`** aplica la funció step a tots els valors alhora.

### 4.2 Què fa, geomètricament

Aquí ve la idea que ho explica tot. L'equació $w_1 x_1 + w_2 x_2 + b = 0$ és
**una recta**. Tots els punts a un costat de la recta donen 1 i tots els de
l'altre costat donen 0. **Un perceptró és una recta que parteix el pla en dos.**

In [ ]:
def step_neuron(weights, bias):
    """Return the prediction function of a single perceptron with fixed weights."""
    return lambda X: np.where(X @ weights + bias >= 0, 1, 0)


y_and = np.array([0, 0, 0, 1])

plot_decision_boundary(step_neuron(weights, bias), X_logic, y_and,
                       title="AND: w = (1, 1), b = -1.5")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

El punt $(1, 1)$, l'únic que ha de donar 1, queda a la zona taronja. Els altres
tres, a la blava.

La línia negra són els punts on $z$ val **exactament 0**, és a dir, on
$x_1 + x_2 = 1{,}5$. Per damunt de la línia $z$ és positiu i la neurona diu 1; per
davall és negatiu i diu 0. Els **pesos** decideixen la inclinació de la recta i el
**biaix** la desplaça.

Si canvies el biaix a $-0{,}5$, la recta es desplaça i la neurona es converteix
en una porta OR. Prova-ho: és un canvi d'un sol número.

### 4.3 Com aprèn: la regla del perceptró

Fins aquí hem triat els pesos a mà. El que va aportar Rosenblatt és un
**procediment per trobar-los sols**, i és sorprenentment senzill. Explicat amb
paraules:

1. Es comença amb uns pesos qualssevol.
2. Es passa per les mostres d'entrenament una a una. Per a cada mostra, el
   perceptró fa la seva predicció.
3. **Si encerta, no toca res.**
4. **Si havia de dir 1 i ha dit 0** (la $z$ era massa baixa), a cada pes li
   **suma** una mica de la seva entrada, i al biaix també. Així la $z$ d'aquesta
   mostra serà una mica més gran la propera vegada. **Si havia de dir 0 i ha dit
   1** (la $z$ era massa alta), **resta** en lloc de sumar.

Cada correcció mou la recta una mica cap al costat bo per a la mostra que s'ha
equivocat. Una entrada que val 0 no ha contribuït a l'error, i el seu pes no es
toca. Una **epoch** és una passada completa per totes les mostres, i es fan
epochs fins que ja no s'equivoca amb cap mostra o fins a un màxim.

Vegem-ho a mà amb la porta AND. Comencem amb $w_1 = 0$, $w_2 = 0$, $b = 0$, cada
correcció és de 0,1 i la predicció és 1 si $z \geq 0$. La primera epoch:

| Mostra | Resposta | $z$ | Predicció | Què passa | Pesos després |
|---|---|---|---|---|---|
| (0, 0) | 0 | 0 | 1 | Error. Les entrades valen 0: només baixa el biaix | $w_1 = 0$, $w_2 = 0$, $b = -0{,}1$ |
| (0, 1) | 0 | −0,1 | 0 | Encerta | sense canvis |
| (1, 0) | 0 | −0,1 | 0 | Encerta | sense canvis |
| (1, 1) | 1 | −0,1 | 0 | Error. Pugen els dos pesos i el biaix | $w_1 = 0{,}1$, $w_2 = 0{,}1$, $b = 0$ |

Dos errors a la primera epoch. A la segona es repeteix el mateix amb els pesos
nous, i cada error torna a moure la recta una mica.

El **learning rate** (el 0,1 de l'exemple) diu com de grans són aquestes empentes. És el
primer **hiperparàmetre** que trobem: un valor que no s'aprèn, sinó que triem
nosaltres abans d'entrenar.

No cal programar-ho: **scikit-learn té el perceptró de Rosenblatt** a
`sklearn.linear_model.Perceptron`, amb la mateixa interfície que els models del
5134. Els paràmetres que farem servir són:

- `eta0`: el learning rate (per defecte val 1).
- `max_iter`: el nombre màxim d'epochs.
- `shuffle`: si barreja l'ordre de les mostres a cada epoch. El posarem a `False`
  perquè el resultat sigui fàcil de seguir.
- `random_state`: la llavor, perquè cada execució doni el mateix resultat.

Normalment farem `fit`, que fa totes les epochs de cop. Però ara volem **veure
l'entrenament epoch a epoch**, i per això farem servir `partial_fit`, que en fa
**una sola** cada vegada que el crides. La funció següent el crida `n_epochs`
vegades i, després de cada epoch, apunta quantes mostres encara classifica
malament i quins pesos té.

In [ ]:
def train_by_epoch(model, X, y, n_epochs):
    """Train `model` one epoch at a time with partial_fit and record its progress.

    Returns the number of misclassified samples and the (weights, bias) after each epoch.
    """
    errors, history = [], []
    for _ in range(n_epochs):
        model.partial_fit(X, y, classes=np.array([0, 1]))
        errors.append(int(np.sum(model.predict(X) != y)))
        history.append((model.coef_.ravel().copy(), model.intercept_[0]))
    return errors, history

Els pesos que ha après un model de scikit-learn es guarden en atributs que
**acaben amb guió baix**: `coef_` són els pesos i `intercept_` és el biaix.
Recordau el `coef_` de la regressió lineal del 5134: és la mateixa convenció, i
vol dir «això només existeix després d'entrenar».

Entrenem el perceptró amb la taula de la porta AND. Ara **no li diem els pesos**:
li donem les quatre files i les respostes, i que els trobi.

In [ ]:
perceptron_and = Perceptron(eta0=0.1, shuffle=False, random_state=42)
errors_and, history_and = train_by_epoch(perceptron_and, X_logic, y_and, n_epochs=8)

print("Errors per epoch:", errors_and)
print("Learned weights :", perceptron_and.coef_.ravel().round(3))
print("Learned bias    :", perceptron_and.intercept_.round(3))
print("Predictions     :", perceptron_and.predict(X_logic), " expected:", y_and)

Mira la llista d'errors per epoch: després de la primera epoch encara s'equivoca
en 2 de les 4 mostres, després de la segona en 1, i a partir de la tercera ja no
s'equivoca mai més. A partir d'aquí els pesos ja no es mouen.

Els pesos que ha trobat no són $(1, 1, -1{,}5)$ com els nostres, però **fan la
mateixa feina**. Hi ha infinites rectes que separen aquests punts, i el perceptró
s'atura a la primera que troba.

Vegem-ho: dibuixem la recta al final de cada una de les primeres epochs.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8), sharex=True, sharey=True)

for epoch, ax in enumerate(axes, start=1):
    w, b = history_and[epoch - 1]
    plot_decision_boundary(step_neuron(w, b), X_logic, y_and, ax=ax,
                           title=f"after epoch {epoch}: {errors_and[epoch - 1]} errors")

fig.suptitle("The perceptron learning AND", fontsize=14)
plt.tight_layout()
plt.show()

Al principi, amb els pesos aleatoris, la recta és a qualsevol lloc. A cada epoch
es va desplaçant i girant fins que deixa el punt $(1, 1)$ tot sol a un costat.

Rosenblatt va demostrar un resultat important, el **teorema de convergència del
perceptró**: **si existeix una recta que separa les dues classes, el perceptró
la trobarà en un nombre finit de passos.** Guardau-vos la condició, perquè és el
que farà caure tot a la secció 5.

### 4.4 Amb dades reals: pingüins

Les portes lògiques tenen quatre punts. Provem-ho amb els pingüins del 5134:
volem distingir **Adelie** de **Gentoo** a partir de dues mesures, la **fondària
del bec** i la **llargada de l'aleta**.

In [ ]:
penguins = pd.read_csv(URL_DATA)

FEATURES = ["bill_depth_mm", "flipper_length_mm"]
two_species = penguins[penguins["species"].isin(["Adelie", "Gentoo"])].dropna(subset=FEATURES)

X_peng = two_species[FEATURES].to_numpy()
y_peng = (two_species["species"] == "Gentoo").astype(int).to_numpy()   # 1 = Gentoo, 0 = Adelie

print(f"{len(X_peng)} penguins: {np.sum(y_peng == 0)} Adelie and {np.sum(y_peng == 1)} Gentoo")

plt.scatter(X_peng[:, 0], X_peng[:, 1], c=y_peng, cmap="coolwarm", edgecolor="k", s=35)
plt.xlabel("bill depth (mm)")
plt.ylabel("flipper length (mm)")
plt.title("Adelie (blue) vs Gentoo (red)")
plt.show()

Els dos grups es veuen clarament separats: els Gentoo tenen l'aleta més llarga i
el bec menys fondo. **Hi cap una recta entre els dos núvols**, i per tant el
perceptró ha de poder aprendre-la.

Partim en entrenament i prova, com al 5134, i entrenem **dues vegades**: una amb
les mesures tal com vénen i una altra amb les mesures **escalades** amb
`StandardScaler`, que resta la mitjana i divideix per la desviació típica de cada
columna. Ja el coneixeu del 5134.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_peng, y_peng, test_size=0.2, random_state=42, stratify=y_peng)

# The scaler learns the mean and std from the training set only
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)

errors_raw, _ = train_by_epoch(Perceptron(eta0=0.1, shuffle=False, random_state=42),
                               X_train, y_train, n_epochs=10)
errors_scaled, _ = train_by_epoch(Perceptron(eta0=0.1, shuffle=False, random_state=42),
                                  X_train_scaled, y_train, n_epochs=10)

print(f"{len(X_train)} training penguins")
print("Errors per epoch, raw data   :", errors_raw)
print("Errors per epoch, scaled data:", errors_scaled)

Tots dos acaben sense errors, però mira quant costa. Sense escalar, el perceptró
es passa les dues primeres epochs equivocant-se en 98 dels 219 pingüins, gairebé
la meitat. Escalat, ho resol pràcticament a la primera.

El motiu és la regla d'aprenentatge: cada empenta és proporcional al valor de
l'entrada. L'aleta mesura uns 200 mm i el bec uns 17 mm, així que **les correccions
del pes de l'aleta són deu vegades més grans** que les del bec. La recta pega
bandades enormes en una direcció i avança a poc a poc en l'altra.

Amb les dues columnes a la mateixa escala, les empentes són equilibrades. **A les
xarxes neuronals escalar les entrades no és opcional**, i ho repetirem cada
vegada. Al 5134 ho treballareu a fons a la UT3.

Fixa't també que l'escalador s'ajusta **només amb entrenament**. Si féssim servir
el test, el model tindria informació de dades que suposadament no ha vist mai.

Per no haver-nos de recordar d'escalar cada vegada, ho empaquetem en una
**pipeline**: escalador i perceptró units en un sol model. Quan fem `fit`,
l'escalador aprèn amb les dades d'entrenament; quan fem `predict` o `score`, les
dades noves s'escalen automàticament amb aquells mateixos valors.

In [ ]:
penguin_model = make_pipeline(StandardScaler(), Perceptron(eta0=0.1, shuffle=False, random_state=42))
penguin_model.fit(X_train, y_train)

print(f"Accuracy on test: {penguin_model.score(X_test, y_test):.3f}")

plot_decision_boundary(penguin_model.predict, X_test, y_test, margin=5,
                       title="Perceptron on test penguins")
plt.xlabel("bill depth (mm)")
plt.ylabel("flipper length (mm)")
plt.show()

Encerta tots els pingüins de prova. Un perceptró, que és una sola neurona amb
tres números apresos, és suficient per a aquest problema, perquè **és linealment
separable**.

Fixa't que el gràfic ja és en mil·límetres: la pipeline escala per dins, i
nosaltres treballem sempre amb les dades originals.

## 5. 1969: el mur de la XOR

La premsa de l'època va rebre el perceptró amb un entusiasme desbocat. Es va
arribar a publicar que seria el germen d'ordinadors capaços de caminar, parlar,
veure i ser conscients de la seva existència.

L'any 1969, Marvin Minsky i Seymour Papert, del MIT, van publicar el llibre
*Perceptrons*, on analitzaven amb rigor matemàtic què podia fer i què no un
perceptró. I l'exemple més senzill del que **no** podia fer era la porta **XOR**
(o exclusiva): la sortida és 1 quan **una i només una** de les entrades és 1.

| x1 | x2 | XOR |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Donem-li al perceptró moltes epochs per si de cas.

In [ ]:
y_xor = np.array([0, 1, 1, 0])

perceptron_xor = Perceptron(eta0=0.1, shuffle=False, random_state=42)
errors_xor, _ = train_by_epoch(perceptron_xor, X_logic, y_xor, n_epochs=50)

print("Errors per epoch (first 15):", errors_xor[:15])
print("Predictions:", perceptron_xor.predict(X_logic), " expected:", y_xor)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(range(1, 51), errors_xor, marker="o", markersize=3)
ax1.set_xlabel("epoch")
ax1.set_ylabel("misclassified samples")
ax1.set_ylim(-0.2, 4.5)
ax1.set_title("It never reaches 0 errors")
plot_decision_boundary(perceptron_xor.predict, X_logic, y_xor, ax=ax2,
                       title="Final boundary: still wrong")
plt.tight_layout()
plt.show()

No hi ha manera. Els errors no arriben mai a zero: al final de cada epoch el
perceptró s'equivoca en 2 de les 4 mostres, i acaba dient 0 a tot. Els pesos van
i vénen indefinidament, perquè cada correcció que arregla un punt n'espatlla un
altre.

**No és un problema d'epochs ni del learning rate.** Mira els
quatre punts: els dos vermells són a una diagonal i els dos blaus a l'altra.
**No existeix cap recta que deixi els vermells a un costat i els blaus a l'altre.**
El problema no és linealment separable, i per tant el teorema de convergència no
ens cobreix.

La XOR és un exemple de joguina, però representa una limitació enorme: moltes
relacions del món real no es poden separar amb una recta. El llibre de Minsky i
Papert va contribuir a una forta retallada del finançament de la recerca en xarxes
neuronals durant els anys setanta. En diem el **primer hivern de la IA**.

### Pausa: com ho resoldria un algorisme clàssic?

Aquí és on s'ha d'entendre la diferència entre les dues famílies.

Al 5134, quan un model lineal no pot amb unes dades, la solució és **fabricar
noves característiques** a mà: termes polinòmics a la UT2, transformacions a la
UT3. Provem-ho. Una persona que mira la taula de la XOR pot tenir una idea: afegir
una tercera columna amb el producte $x_1 \cdot x_2$, que només val 1 a la fila
$(1, 1)$.

In [ ]:
# Feature engineering by hand: add the product x1·x2 as a third column
X_xor_engineered = np.column_stack([X_logic, X_logic[:, 0] * X_logic[:, 1]])
print(X_xor_engineered)

perceptron_eng = Perceptron(eta0=0.1, shuffle=False, random_state=42)
errors_eng, _ = train_by_epoch(perceptron_eng, X_xor_engineered, y_xor, n_epochs=50)

print("\nErrors per epoch (first 15):", errors_eng[:15])
print("Predictions:", perceptron_eng.predict(X_xor_engineered), " expected:", y_xor)

Amb la columna nova, **el mateix perceptró ho resol**: després d'uns quants
vaivens, a partir de l'epoch 9 ja no s'equivoca mai més. En tres dimensions sí que
existeix un pla que separa els punts.

Però algú ha hagut de **pensar** que el producte era la característica que calia.
Amb quatre punts és fàcil. Amb una fotografia de 1.000 × 1.000 píxels, quina
columna nova hauríem d'inventar per saber si hi surt un gat?

Aquesta és la pregunta a la qual responen les xarxes neuronals:

> **I si la xarxa aprengués ella mateixa quines característiques noves necessita?**

## 6. La solució: afegir capes

La idea que desbloqueja la XOR és **posar neurones entre l'entrada i la sortida**.
Ho farem primer a mà, amb pesos triats per nosaltres, per veure per què funciona.

La XOR es pot escriure com una combinació de portes que un perceptró sí que sap
fer: **XOR = (x1 OR x2) AND NOT (x1 AND x2)**. És a dir:

- una neurona fa **OR**,
- una altra fa **NAND** (el contrari d'AND),
- i una tercera fa **AND** de les dues anteriors.

```
  x1 ──┬──▶ [ OR   ] ──┐
       │               ├──▶ [ AND ] ──▶ XOR
  x2 ──┴──▶ [ NAND ] ──┘
   entrada   capa oculta       sortida
```

Les dues neurones del mig formen una **capa oculta**: no les veiem ni a l'entrada
ni a la sortida. Hem construït el nostre primer **perceptró multicapa**.

In [ ]:
def step(z):
    return np.where(z >= 0, 1, 0)


def layer(X, weights, bias):
    """One layer of step neurons: weighted sum followed by the step function."""
    return step(X @ weights + bias)


# Hidden layer: 2 neurons (columns of the weight matrix)
W_hidden = np.array([[1.0, -1.0],     # weights from x1 to (OR, NAND)
                     [1.0, -1.0]])    # weights from x2 to (OR, NAND)
b_hidden = np.array([-0.5, 1.5])      # biases of (OR, NAND)

# Output layer: 1 neuron doing AND of the two hidden neurons
W_output = np.array([1.0, 1.0])
b_output = -1.5

H = layer(X_logic, W_hidden, b_hidden)
xor_out = layer(H, W_output, b_output)

print("x1 x2 | h1(OR) h2(NAND) | output")
print("------+-----------------+-------")
for (x1, x2), (h1, h2), out in zip(X_logic.astype(int), H, xor_out):
    print(f" {x1}  {x2} |   {h1}      {h2}     |   {out}")

Funciona: la sortida és exactament la XOR.

Fixa't que hem passat de vectors de pesos a **matrius de pesos**. `W_hidden` té
una fila per entrada i una columna per neurona oculta, i `X_logic @ W_hidden`
calcula les dues neurones per a les quatre mostres d'un sol cop. Així es programa
una capa sencera, tingui 2 neurones o 2.000.

Ara ve el més important de tot el notebook. Dibuixem els quatre punts **a l'espai
original** $(x_1, x_2)$ i **a l'espai de la capa oculta** $(h_1, h_2)$, que és el
que veu la neurona de sortida.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

plot_decision_boundary(lambda X: np.zeros(len(X)), X_logic, y_xor, ax=ax1,
                       title="Input space (x1, x2): no line separates them")
ax1.set_xlabel("x1")
ax1.set_ylabel("x2")

# Tiny random jitter only so that the two points that land on (1, 1) are both visible
jitter = np.array([[0, 0], [-0.04, 0.04], [0.04, -0.04], [0, 0]])
plot_decision_boundary(step_neuron(W_output, b_output), H + jitter, y_xor, ax=ax2,
                       title="Hidden space (h1, h2): now a line does")
ax2.set_xlabel("h1 = OR(x1, x2)")
ax2.set_ylabel("h2 = NAND(x1, x2)")

plt.tight_layout()
plt.show()

A l'esquerra, el problema impossible. A la dreta, **els mateixos quatre punts
vistos per la capa oculta**: els dos punts vermells (classe 1) cauen tots dos a
$(1, 1)$ i els blaus (classe 0) queden a les cantonades. Ara una recta els separa
sense problema, i la neurona de sortida només ha de fer el que ja sabia fer un
perceptró.

> **La capa oculta no resol el problema: el transforma en un problema que una
> sola neurona sap resoldre.**

Això és exactament el que fèiem a la secció 5 afegint la columna $x_1 \cdot x_2$.
La capa oculta ha **creat noves característiques**, $h_1$ i $h_2$. La diferència
és que aquí les hem triades nosaltres, però **en una xarxa entrenada les triarà
l'entrenament**. L'article de 1986 que va popularitzar la backpropagation es titulava,
precisament, *Learning representations by back-propagating errors*: aprendre
representacions.

### Llavors, per què es van quedar encallats fins al 1986?

Perquè saber que les capes funcionen no diu **com trobar els pesos**. Hi havia
dos obstacles:

1. **El problema d'assignar la culpa.** La regla del perceptró corregeix una
   neurona comparant la seva sortida amb la resposta correcta. Però per a una
   neurona oculta **no hi ha resposta correcta**: ningú no ens diu que $h_1$ havia
   de fer OR. Si la xarxa s'equivoca, quina neurona oculta en té la culpa, i
   quanta?
2. **La funció step és plana.** Si modifiques un pes una mica, la sortida
   normalment no canvia gens (continua sent 0 o 1). No hi ha manera de saber cap a
   on s'ha de moure el pes per millorar.

La solució té tres peces: **funcions d'activació sense salts** (secció 7), una
**funció de loss i el gradient descent** (secció 8) i la
**backpropagation** (secció 9).

## 7. Funcions d'activació sense salts

El primer obstacle es resol substituint la funció step per una funció que faci
**pràcticament el mateix, però pujant de mica en mica, sense salts**. Així, un petit canvi en un
pes produeix un petit canvi a la sortida, i podem saber si anem bé o malament.

Dibuixem les funcions d'activació més habituals i, a sota, el seu **pendent**:
quant canvia la sortida quan l'entrada canvia una mica.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def relu(z):
    return np.maximum(0, z)


z = np.linspace(-5, 5, 400)

activations = {
    "step":    np.where(z >= 0, 1, 0),
    "sigmoid": sigmoid(z),
    "tanh":    np.tanh(z),
    "ReLU":    relu(z),
}

fig, axes = plt.subplots(2, 4, figsize=(14, 6), sharex=True)
for col, (name, value) in enumerate(activations.items()):
    slope = np.round(np.gradient(value, z), 6)   # measured numerically: change in output / change in z
    slope[np.abs(slope) > 5] = np.nan            # hide the vertical jump of the step at z = 0
    axes[0, col].plot(z, value, linewidth=2.5)
    axes[0, col].set_title(name)
    axes[1, col].plot(z, slope, linewidth=2.5, color="tab:orange")
    axes[1, col].set_xlabel("z")
axes[0, 0].set_ylabel("output")
axes[1, 0].set_ylabel("slope")
plt.tight_layout()
plt.show()

El pendent no l'hem calculat amb cap fórmula: `np.gradient` el **mesura**
comparant cada punt de la corba amb els seus veïns, que és exactament el que
vol dir pendent.

Llegeix les dues files alhora:

- **Step.** El pendent és **zero a tot arreu** (i infinit just al 0). No dóna
  cap pista de cap a on moure els pesos. Per això no es pot fer servir per
  entrenar capes.
- **Sigmoide.** La versió sense salts de la funció step: va de 0 a 1 i la sortida es pot
  llegir com una **probabilitat**. És la que fa servir `MLPClassifier` a la neurona
  de sortida quan classifiquem entre dues classes.
- **Tangent hiperbòlica (tanh).** La mateixa forma, però va de −1 a 1 i està
  centrada en el zero, cosa que fa que les capes ocultes aprenguin més de pressa.
  És la que farem servir a la capa oculta en aquest notebook (`activation="tanh"`).
- **ReLU.** Deixa passar els positius i talla els negatius. És tan simple que
  sembla que no hauria de funcionar, però és **l'opció per defecte de les xarxes
  profundes actuals**. Veurem per què en una unitat posterior, quan les xarxes
  tinguin moltes capes.

Una observació que us serà útil al 5134: **una neurona amb sigmoide és exactament
una regressió logística**, que veureu a la UT4. La regressió logística és una
neurona. El que la converteix en xarxa neuronal és **apilar-ne moltes en capes**.

## 8. Mesurar l'error i baixar la vall: el gradient descent

Per entrenar una xarxa necessitem dues coses:

1. **Un número que digui com de malament ho fa**: la **loss** (*loss function*).
   Com més petita, millor.
2. **Una manera de modificar els pesos perquè aquest número baixi.**

A les diapositives de la UT1 del 5134 vau veure que, si proves molts valors d'un
coeficient i mesures l'error de cadascun, **l'error dibuixa una vall**, i entrenar
és trobar-ne el fons. Però una xarxa pot tenir milions de pesos: no podem provar
valors un per un.

La solució és el **gradient descent**. Imagina que ets a la muntanya enmig
d'una boira tan espessa que només veus els teus peus. Per baixar a la vall, mires
**cap a on fa pendent el terra** i fas una passa cap avall. I repeteixes.

Ho veurem amb el cas més petit possible: un sol pes $w$ i unes dades que segueixen
aproximadament $y = 2x$. La loss serà l'error quadràtic mitjà.

In [ ]:
rng = np.random.default_rng(0)
x_toy = rng.uniform(-1, 1, 50)
y_toy = 2 * x_toy + rng.normal(0, 0.2, 50)


def toy_loss(w):
    """Mean squared error of the model y = w·x."""
    return np.mean((w * x_toy - y_toy) ** 2)


def toy_slope(w, step=1e-4):
    """Slope of the valley at w: take a tiny step to each side and compare the loss."""
    return (toy_loss(w + step) - toy_loss(w - step)) / (2 * step)


w_values = np.linspace(-1, 5, 200)
plt.plot(w_values, [toy_loss(w) for w in w_values], linewidth=2.5)
plt.ylim(0, 4)
plt.xlabel("w")
plt.ylabel("loss (MSE)")
plt.title("The loss draws a valley")
plt.show()

El fons de la vall és a prop de $w = 2$, com esperàvem.

La funció `toy_slope` fa el que faries amb els ulls tancats a la muntanya: fa una
passa minúscula a cada costat i compara on és més alt. Si la loss és més gran a
la dreta, el pendent és positiu i per baixar hem d'anar cap a l'esquerra. Si és
negatiu, al revés. Per això la regla del gradient descent **resta** el pendent:

$$w \leftarrow w - \text{learning\_rate} \cdot \text{pendent}$$

En una xarxa amb molts pesos, el conjunt de tots els pendents s'anomena
**gradient**, i d'aquí el nom. L'algorisme és el mateix per a un pes que per a un
milió.

Posem-lo en marxa des de $w = -0{,}5$ amb quatre coeficients d'aprenentatge diferents.

In [ ]:
def gradient_descent(w_start, learning_rate, n_steps):
    """Run gradient descent on toy_loss and return every visited value of w."""
    path = [w_start]
    w = w_start
    for _ in range(n_steps):
        w = w - learning_rate * toy_slope(w)
        path.append(w)
    return np.array(path)


fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for ax, lr in zip(axes, [0.05, 0.6, 2.7, 3.2]):
    path = gradient_descent(w_start=-0.5, learning_rate=lr, n_steps=12)
    ax.plot(w_values, [toy_loss(w) for w in w_values], linewidth=2, color="lightgray")
    ax.plot(path, [toy_loss(w) for w in path], "o-", color="tab:red", markersize=5)
    ax.set_xlim(-1, 5)
    ax.set_ylim(0, 6)
    ax.set_title(f"learning_rate = {lr}  ->  w after 12 steps = {path[-1]:.2f}")
    ax.set_xlabel("w")
axes[0].set_ylabel("loss")
plt.tight_layout()
plt.show()

Els quatre gràfics tenen la mateixa vall i el mateix punt de partida. L'únic que
canvia és la mida de la passa:

- **Massa petit** (0,05): cada passa és diminuta. Va en bona direcció, però en
  dotze passes encara no ha arribat al fons. Entrenar costaria moltíssim.
- **Adequat** (0,6): arriba al fons en poques passes.
- **Gran** (2,7): cada passa **se'n passa** a l'altre costat de la vall i va
  rebotant d'una paret a l'altra. Acaba arribant, però perdent el temps.
- **Massa gran** (3,2): cada rebot és més alt que l'anterior i la loss, en lloc
  de baixar, **creix**. L'entrenament **divergeix** i el pes se'n va a l'infinit.
  Els punts surten del gràfic.

El learning rate és, probablement, **l'hiperparàmetre més important
d'una xarxa neuronal**, i ho comprovarem a la secció 11.

## 9. Backpropagation: repartir la culpa cap enrere

Ja tenim funcions d'activació sense salts i sabem baixar una vall si coneixem el pendent. Falta
l'última peça: **saber cap a on s'ha de moure cada pes de la xarxa**, inclosos els
de les capes ocultes. Això és la **backpropagation**.

La idea és una **cadena de responsabilitats**:

1. **Pas endavant** (*forward pass*). Les entrades travessen la xarxa capa a capa
   fins a la sortida. Obtenim una predicció.
2. **Loss.** Comparem la predicció amb la resposta correcta.
3. **Pas enrere** (*backward pass*). Comencem per la sortida: sabem exactament
   quant s'ha equivocat. Llavors anem **cap enrere**: cada neurona oculta rep una
   part de la culpa **proporcional al pes que la connecta amb la sortida** (si el
   seu vot pesava molt, té molta culpa) i **a com de sensible era** en aquell
   moment (el pendent de la seva activació, el gràfic de la secció 7).
4. **Actualització.** Amb la culpa de cada neurona sabem cap a on moure cada pes i
   fem una passa de gradient descent.

Pensa en un equip que perd un partit. L'entrenador no renya tothom igual: primer
mira qui ha fallat el darrer pas, després qui li ha passat la pilota i com de
decisiva era la passada, i així cap enrere. Cada jugador corregeix en proporció a
la seva part de culpa.

**Nosaltres no ho programarem.** Aquests quatre passos són exactament el que fa
`MLPClassifier` de scikit-learn cada vegada que entrena. El que sí que farem és
**observar-lo**.

### 9.1 La funció de loss per classificar

Quan la sortida és una probabilitat, la loss habitual és l'**entropia creuada
binària** (*binary cross-entropy* o *log loss*). No cal saber-ne la fórmula; el que
importa és com castiga. scikit-learn la té a `sklearn.metrics.log_loss`:

In [ ]:
from sklearn.metrics import log_loss

print("True class is 1. Loss depending on the predicted probability of class 1:\n")
for p in [0.99, 0.9, 0.7, 0.5, 0.3, 0.1, 0.01]:
    print(f"  predicted {p:4.2f}  ->  loss {log_loss([1], [p], labels=[0, 1]):.3f}")

Si la resposta és 1 i la xarxa diu 0,99, la loss és gairebé zero. Si diu 0,5
(no ho sap), la loss és moderada. Si diu 0,01 (**està segura i s'equivoca**),
la loss es dispara. És exactament el que volem premiar i castigar.

### 9.2 Mirar com aprèn una xarxa, passa a passa

Creem una xarxa **2-3-1** amb `MLPClassifier`: 2 entrades (les columnes de `X`),
**3 neurones ocultes** i 1 neurona de sortida. Els paràmetres que farem servir
durant tot el notebook són:

- `hidden_layer_sizes=(3,)`: una capa oculta de 3 neurones. Una tupla, perquè en
  pot haver més d'una: `(16, 16)` serien dues capes de 16.
- `activation="tanh"`: la funció d'activació de la capa oculta.
- `learning_rate_init`: el learning rate.
- `max_iter`: el nombre màxim d'epochs.
- `random_state`: la llavor dels pesos inicials.

Igual que amb el perceptró, farem servir `partial_fit` per fer **una sola passa**
d'entrenament cada vegada i mirar la loss, que scikit-learn desa a l'atribut
`loss_`.

In [ ]:
step_by_step = MLPClassifier(hidden_layer_sizes=(3,), activation="tanh",
                             learning_rate_init=0.05, random_state=42)

for step in range(1, 11):
    step_by_step.partial_fit(X_logic, y_xor, classes=[0, 1])
    print(f"after step {step:2d}: loss = {step_by_step.loss_:.4f}")

print("\nShapes of the learned weights:", [W.shape for W in step_by_step.coefs_])

A cada passa la loss baixa una mica. **Entrenar una xarxa neuronal és repetir
aquests quatre passos milers de vegades.** No hi ha res més. Les xarxes que
reconeixen imatges o generen text fan aquest mateix bucle, amb més capes, més
neurones i més dades.

Mira també les formes dels pesos (`coefs_`), perquè diuen com és la xarxa:

- **(2, 3)**: de 2 entrades a 3 neurones ocultes. Una fila per entrada i una
  columna per neurona: **6 pesos**.
- **(3, 1)**: de 3 neurones ocultes a 1 sortida: **3 pesos** més.

Cada neurona té, a més, el seu biaix, a `intercepts_`. La xarxa sencera té, doncs,
6 + 3 + 3 + 1 = 13 números per aprendre. Les xarxes grans en tenen milions, però
s'entrenen igual.

## 10. L'MLP aprèn la XOR tot sol

Ara sí. Li donem a la xarxa les quatre files de la XOR i **no li diem res
més**: ni que calen portes OR i NAND, ni quins pesos fan falta.

In [ ]:
mlp_xor = MLPClassifier(hidden_layer_sizes=(3,), activation="tanh", learning_rate_init=0.05,
                        max_iter=3000, tol=1e-6, n_iter_no_change=50, random_state=42)
mlp_xor.fit(X_logic, y_xor)

print("Probabilities:", mlp_xor.predict_proba(X_logic)[:, 1].round(3))
print("Predictions  :", mlp_xor.predict(X_logic), " expected:", y_xor)
print("Epochs used  :", mlp_xor.n_iter_)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(mlp_xor.loss_curve_)
ax1.set_xlabel("epoch")
ax1.set_ylabel("binary cross-entropy")
ax1.set_title("Loss curve")
plot_decision_boundary(mlp_xor.predict, X_logic, y_xor, ax=ax2, title="Learned boundary")
plt.tight_layout()
plt.show()

Ho ha après. `predict_proba` dóna la probabilitat de cada classe; la segona
columna és la de la classe 1, i `predict` només hi aplica el llindar de 0,5.

La **corba de loss** (`loss_curve_`) és el primer gràfic que s'ha de mirar sempre
que s'entrena una xarxa: comença alta, cau durant les primeres epochs i després
s'aplana a prop de zero. Quan una corba s'aplana, continuar entrenant ja no aporta
gaire, i per això scikit-learn s'atura sol: els paràmetres `tol` i
`n_iter_no_change` li diuen que pari si la loss no millora durant 50 epochs
seguides. `n_iter_` diu quantes epochs ha fet realment.

I mira la frontera: **ja no és una recta**. Són dues rectes paral·leles que formen
una franja diagonal: els punts blaus queden a dins i els vermells, a fora. Això és
el que no podia fer el perceptró.

### Què ha après la capa oculta?

A la secció 6 vam veure que la capa oculta transformava l'espai. Ara que els pesos
els ha trobat l'entrenament, mirem què ha fet. Entrenem una xarxa amb **només 2
neurones ocultes**, perquè el seu espai es pugui dibuixar en dues dimensions, i
provem **dues llavors diferents**.

Per veure què surt de la capa oculta fem el mateix que vam fer a mà a la secció
6, però amb els pesos apresos: suma ponderada amb `coefs_[0]` i `intercepts_[0]` i
després la tanh.

In [ ]:
def hidden_activations(model, X):
    """What the output neuron 'sees': the values of the first hidden layer."""
    return np.tanh(X @ model.coefs_[0] + model.intercepts_[0])


fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, seed in zip(axes, [1, 0]):
    model = MLPClassifier(hidden_layer_sizes=(2,), activation="tanh", learning_rate_init=0.05,
                          max_iter=3000, tol=1e-6, n_iter_no_change=50,
                          random_state=seed).fit(X_logic, y_xor)
    H_learned = hidden_activations(model, X_logic)
    # Decreasing marker sizes: if two points land on the same spot, both remain visible
    ax.scatter(H_learned[:, 0], H_learned[:, 1], c=y_xor, cmap="coolwarm", edgecolor="k",
               s=[320, 220, 130, 60])
    offsets = [(10, 10), (10, -16), (-44, 10), (-44, -16)]   # so overlapping labels stay readable
    for (h1, h2), (x1, x2), offset in zip(H_learned, X_logic.astype(int), offsets):
        ax.annotate(f"({x1},{x2})", (h1, h2), textcoords="offset points", xytext=offset)
    ax.set_xlim(-1.2, 1.2)
    ax.set_ylim(-1.2, 1.2)
    ax.set_xlabel("hidden neuron 1")
    ax.set_ylabel("hidden neuron 2")
    ax.set_title(f"random_state={seed}: predicts {model.predict(X_logic)}, "
                 f"final loss {model.loss_curve_[-1]:.3f}")

plt.tight_layout()
plt.show()

Dues lliçons en un sol gràfic.

**Amb `random_state=1` la xarxa ha fet el mateix que nosaltres a mà.** Ha recol·locat
els quatre punts de manera que els dos vermells i els dos blaus queden separables
amb una recta. No sabem si les seves neurones fan exactament OR i NAND, i no cal:
han trobat **la seva pròpia representació**. Aquesta és la idea que diferencia les
xarxes neuronals.

**Amb `random_state=0`, la mateixa xarxa, amb les mateixes dades i el mateix codi,
s'ha encallat.** La loss s'ha aturat lluny de zero i una de les quatre
prediccions és incorrecta. Mira el gràfic: la capa oculta ha enviat el punt vermell
$(0, 1)$ **al mateix racó que els dos blaus**. A partir d'aquí, cap neurona de
sortida el pot distingir d'ells. El gradient descent ha baixat a una vall que no és la més
profunda: un **mínim local**.

Això és una diferència important amb molts algorismes clàssics. Una regressió
lineal dóna **sempre el mateix resultat** amb les mateixes dades. Una xarxa
neuronal **depèn dels pesos inicials**. A la pràctica es combat amb xarxes més
amples del mínim imprescindible i provant diverses inicialitzacions. En tens un
exercici al final.

## 11. Un problema de veritat: les dues llunes

La XOR són quatre punts. Provem amb un conjunt més realista: **dues llunes**
entrellaçades amb soroll, generat per scikit-learn. És el problema clàssic per
veure si un model és capaç de fer fronteres corbes.

In [ ]:
from sklearn.datasets import make_moons

X_moons, y_moons = make_moons(n_samples=600, noise=0.25, random_state=42)
Xm_train, Xm_test, ym_train, ym_test = train_test_split(
    X_moons, y_moons, test_size=0.25, random_state=42, stratify=y_moons)

plt.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap="coolwarm", edgecolor="k", s=25)
plt.title(f"Two moons: {len(Xm_train)} train, {len(Xm_test)} test")
plt.show()

Cap recta pot separar bé les dues llunes. Enfrontem el perceptró amb un MLP de 8
neurones ocultes.

Aquestes dades ja estan centrades al voltant del zero i tenen escales semblants a
les dues columnes, així que en aquest cas concret no cal escalar.

In [ ]:
perceptron_moons = Perceptron(random_state=42).fit(Xm_train, ym_train)
mlp_moons = MLPClassifier(hidden_layer_sizes=(8,), activation="tanh", learning_rate_init=0.01,
                          max_iter=3000, random_state=42).fit(Xm_train, ym_train)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
plot_decision_boundary(perceptron_moons.predict, Xm_test, ym_test, ax=ax1,
                       title=f"Perceptron: test accuracy {perceptron_moons.score(Xm_test, ym_test):.2f}")
plot_decision_boundary(mlp_moons.predict, Xm_test, ym_test, ax=ax2,
                       title=f"MLP (8 hidden): test accuracy {mlp_moons.score(Xm_test, ym_test):.2f}")
plt.tight_layout()
plt.show()

El perceptró fa l'única cosa que sap fer, una recta, i s'equivoca en un de cada
sis o set punts de prova. L'MLP dibuixa una frontera en forma de S que segueix les
llunes, i l'encert puja fins al 94%. El 6% restant són sobretot punts que el
soroll ha ficat dins la lluna contrària: cap frontera raonable no els pot encertar.

Fixa't que el codi dels dos models és gairebé idèntic: canvia la classe i els
hiperparàmetres, però `fit` i `score` són els de sempre.

### 11.1 Quantes neurones ocultes?

El nombre de neurones de la capa oculta és el segon hiperparàmetre de la xarxa.
Vegem l'efecte.

Per a aquesta comparació farem servir `solver="lbfgs"`, un altre mètode
d'entrenament que ofereix `MLPClassifier`. Amb conjunts petits com aquest arriba
més a prop del millor resultat possible de cada xarxa, i així la comparació depèn
de la mida de la capa i no de si l'entrenament ha tingut temps d'acabar.

Quan una xarxa arriba a `max_iter` sense haver-se estabilitzat, scikit-learn avisa
amb un `ConvergenceWarning`. És un avís que heu de llegir sempre, perquè sol voler
dir que cal més epochs o escalar les dades. Aquí la xarxa de 32 neurones l'activa,
i ho sabem, així que el silenciem només dins d'aquesta cel·la.

In [ ]:
import warnings

from sklearn.exceptions import ConvergenceWarning

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, n_hidden in zip(axes, [1, 2, 4, 32]):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ConvergenceWarning)   # expected for 32 neurons
        model = MLPClassifier(hidden_layer_sizes=(n_hidden,), activation="tanh", solver="lbfgs",
                              max_iter=3000, random_state=42).fit(Xm_train, ym_train)
    plot_decision_boundary(model.predict, Xm_train, ym_train, ax=ax,
                           title=f"{n_hidden} hidden: train {model.score(Xm_train, ym_train):.2f}"
                                 f" / test {model.score(Xm_test, ym_test):.2f}")
plt.tight_layout()
plt.show()

- **1 neurona oculta**: la frontera amb prou feines es corba. No té capacitat.
- **2 neurones**: una mica millor, però encara es queda curta.
- **4 neurones**: ja segueix la forma de les llunes, i l'encert a test salta fins
  al 93%.
- **32 neurones**: l'encert a entrenament s'enfila gairebé al 100%, però a test
  **baixa**. Mira la frontera: fa ziga-zagues per anar a buscar punts concrets.

Amb més neurones, la xarxa pot dibuixar fronteres més complicades. Això és bo fins
que comença a **memoritzar el soroll** de l'entrenament en lloc d'aprendre la
forma general: l'**overfitting** que veureu a la UT2 del 5134. La pista és sempre
la mateixa: l'encert a entrenament puja i el de test no l'acompanya.

### 11.2 El learning rate

Tornem a l'hiperparàmetre de la secció 8, ara sobre una xarxa de veritat. Per
veure'l net farem servir `solver="sgd"` amb `momentum=0`, que és el **descens del
gradient** tal com l'hem dibuixat a la vall, sense cap millora. Comparem les corbes
de loss amb cinc learning rates diferents, sempre amb 500 epochs. Com que volem veure
les corbes senceres, fins i tot les que no arriben enlloc, tornem a silenciar el
`ConvergenceWarning`.

In [ ]:
for lr in [0.001, 0.01, 0.1, 1.0, 10.0]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=ConvergenceWarning)   # 500 epochs on purpose
        model = MLPClassifier(hidden_layer_sizes=(8,), activation="tanh", solver="sgd", momentum=0,
                              learning_rate_init=lr, max_iter=500, n_iter_no_change=500,
                              random_state=42).fit(Xm_train, ym_train)
    unstable = lr > 5
    plt.plot(model.loss_curve_, linewidth=1 if unstable else 2.5, alpha=0.45 if unstable else 1,
             zorder=1 if unstable else 3, label=f"lr={lr}: test {model.score(Xm_test, ym_test):.2f}")

plt.xlabel("epoch")
plt.ylabel("training loss")
plt.ylim(0, 0.8)
plt.legend(loc="upper right", framealpha=1)
plt.title("Same network, different learning rates")
plt.show()

Exactament el que vam veure amb la vall d'un sol pes:

- **0,001**: la loss baixa tan a poc a poc que en 500 epochs gairebé no s'ha
  mogut, i l'encert es queda en un 64%.
- **0,01 i 0,1**: baixa de manera neta, però lenta. Amb 0,01 l'encert encara es
  queda al nivell d'una recta; la xarxa no ha tingut temps d'aprendre la corba.
- **1**: baixa ràpid i arriba a la loss més baixa. En aquesta xarxa petita, un
  coeficient alt encara s'aguanta.
- **10**: la corba és plena de pics. Cada passa se'n passa de llarg i la loss no
  s'estabilitza mai. L'encert final pot semblar bo, però és pura sort: depèn de
  l'epoch exacta en què s'atura l'entrenament.

Llegir la corba de loss per decidir si el learning rate és massa petit o massa gran
és una habilitat que fareu servir durant tot el mòdul.

## 12. Tots els models en una taula

Posem en una taula tots els models que hem fet servir sobre les llunes. Com
sempre al 5134, comencem pel **model de referència** (`DummyClassifier`), que
prediu sempre la classe més freqüent: sense aquest punt de comparació, un
percentatge d'encert no vol dir res.

In [ ]:
import time

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

models = {
    "Dummy (reference)":           DummyClassifier(),
    "Logistic regression":         LogisticRegression(),
    "Perceptron":                  Perceptron(random_state=42),
    "MLP (8 hidden, tanh)":        MLPClassifier(hidden_layer_sizes=(8,), activation="tanh",
                                                 learning_rate_init=0.01, max_iter=3000, random_state=42),
    "MLP (16, 16 hidden, relu)":   MLPClassifier(hidden_layer_sizes=(16, 16), max_iter=2000, random_state=42),
}

rows = []
for name, model in models.items():
    start = time.perf_counter()
    model.fit(Xm_train, ym_train)
    elapsed = time.perf_counter() - start
    rows.append({"model": name,
                 "train accuracy": model.score(Xm_train, ym_train),
                 "test accuracy": model.score(Xm_test, ym_test),
                 "fit time (s)": elapsed})

pd.DataFrame(rows).set_index("model").round(3)

Hi ha molt per llegir en aquesta taula.

**El bucle és idèntic per a tots els models.** Tots comparteixen `fit` i `score`, i
el codi no ha de saber quin és quin. Això és el que vol dir que una biblioteca
tingui una **interfície coherent**: podem intercanviar un component per un altre
sense tocar res més.

**La regressió logística i el perceptró queden per sota.** Tots dos són una sola
neurona i dibuixen una recta. Cap recta no passa d'aquí: no és que la regressió
logística sigui «dolenta», és que aquest problema no és lineal.

**Els MLP guanyen clarament.** El segon té **dues capes ocultes** de 16 neurones
(`hidden_layer_sizes=(16, 16)`) i fa servir l'activació per defecte de
scikit-learn, la ReLU.

Per defecte, `MLPClassifier` no fa servir el gradient descent pur de la
secció 11.2, sinó una variant més sofisticada anomenada **Adam**, que ajusta la
mida de la passa per a cada pes, i treballa amb **batches** de mostres
(*mini-batches*) en lloc de tot el conjunt alhora. La idea de fons és la mateixa:
pas endavant, loss, pas enrere, actualització.

**Mira el temps.** La regressió logística s'entrena en un parell de mil·lisegons.
Les xarxes tarden moltes vegades més. Amb 450 punts no importa; amb milions, és la
diferència entre minuts i dies, i és el motiu pel qual les xarxes grans s'entrenen
amb GPU.

### 12.1 I per a regressió?

Tot el que hem fet és classificació, però una xarxa neuronal també pot predir un
número. Només canvien dues peces: **la neurona de sortida no porta activació**
(ha de poder donar qualsevol valor, no només entre 0 i 1) i **la loss és l'error
quadràtic mitjà** en lloc de l'entropia creuada. scikit-learn ho té a
`MLPRegressor`, amb els mateixos paràmetres que `MLPClassifier`.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor

rng = np.random.default_rng(42)
x_reg = np.sort(rng.uniform(-3, 3, 200)).reshape(-1, 1)
y_reg = np.sin(x_reg).ravel() + rng.normal(0, 0.15, 200)

linear = LinearRegression().fit(x_reg, y_reg)
mlp_reg = MLPRegressor(hidden_layer_sizes=(32,), activation="tanh",
                       learning_rate_init=0.01, max_iter=3000, random_state=42).fit(x_reg, y_reg)

plt.scatter(x_reg, y_reg, s=12, color="gray", alpha=0.6, label="data")
plt.plot(x_reg, linear.predict(x_reg), linewidth=2.5, label=f"LinearRegression  R2={linear.score(x_reg, y_reg):.2f}")
plt.plot(x_reg, mlp_reg.predict(x_reg), linewidth=2.5, label=f"MLPRegressor (32)  R2={mlp_reg.score(x_reg, y_reg):.2f}")
plt.legend()
plt.title("Regression: a line vs a neural network")
plt.show()

La regressió lineal fa l'única cosa que pot fer, una recta. La xarxa, amb una capa
de 32 neurones, **segueix la corba sense que ningú li hagi dit que era una sinusoide**.

Al 5134, per ajustar aquesta corba amb un model lineal, hauríeu de fabricar
característiques polinòmiques (UT2) i triar-ne el grau. Aquí la xarxa ha fet
aquesta feina sola. És, una altra vegada, la idea central d'aquest notebook.

## 13. Resum: algorismes clàssics i xarxes neuronals

| | Algorismes clàssics (5134) | Xarxes neuronals (5149) |
|---|---|---|
| **Què són** | Cada algorisme és una idea concreta: una recta, preguntes encadenades, el marge més ample, els veïns | Moltes neurones iguals i simples, connectades en capes |
| **Característiques** | Les prepara la persona (polinòmiques, transformacions, codificacions) | Les capes ocultes aprenen representacions noves |
| **Entrenament** | Mètodes específics per a cada algorisme; molts donen sempre el mateix resultat | Sempre la mateixa recepta: gradient descent i backpropagation. Depèn de la inicialització |
| **Hiperparàmetres** | Pocs i sovint fàcils d'interpretar | Molts: arquitectura, activacions, learning rate, epochs... |
| **Dades on brillen** | Taules estructurades, de centenars a milions de files | Imatges, so, text i altres dades no estructurades, amb molts exemples |
| **Cost** | Baix: s'entrenen en segons | Alt: sovint necessiten GPU |
| **Interpretabilitat** | Molts es poden llegir (coeficients, decision trees) | Difícils d'explicar: milers o milions de pesos |

Una matisació important perquè no us quedeu amb una idea equivocada: **les xarxes
no són sempre millors**. Amb dades tabulars com les del 5134, un bon random forest o
un *gradient boosting* sovint igualen o superen una xarxa neuronal, s'entrenen molt
més de pressa i s'expliquen millor. Triar bé és part de la feina.

### Les idees que t'has d'endur

1. **Una neurona** fa una suma ponderada de les entrades, hi suma un biaix i hi
   aplica una funció d'activació. Geomètricament, separa l'espai amb una recta.
2. **El perceptró aprèn** corregint els pesos cada vegada que s'equivoca, però
   **només pot resoldre problemes linealment separables**. La XOR no ho és.
3. **Les capes ocultes transformen les dades** en una representació nova on el
   problema sí que és separable. Això és el que les fa diferents.
4. **Per entrenar capes** calen funcions d'activació sense salts, una funció de loss i el
   **gradient descent**, amb un learning rate ben triat.
5. **La backpropagation** reparteix la culpa de l'error cap enrere, capa a capa, per
   saber com s'ha de moure cada pes.

## Exercicis

**Exercici 1. Més portes lògiques.** Entrena el `Perceptron` amb les taules de
**NAND** i **NOR** fent servir `train_by_epoch`. Quantes epochs necessita
cadascuna fins a arribar a 0 errors? Repeteix-ho amb `eta0` de 0,01, 0,1 i 1.
Canvia el nombre d'epochs necessàries? Per què creus que sí o que no? *(Pista: el
perceptró de scikit-learn comença amb tots els pesos a zero. Què passa si
multipliques totes les correccions per 10?)*

**Exercici 2. Pingüins que no se separen.** Repeteix la secció 4.4 amb **Adelie**
contra **Chinstrap**, fent servir `bill_length_mm` i `bill_depth_mm`. Dibuixa primer
les dades. Arriba el perceptró a 0 errors? Què li passa a la llista d'errors al
llarg de 100 epochs? Relaciona-ho amb el teorema de convergència.

**Exercici 3. Barrejar les mostres.** Repeteix l'exercici 2 amb `shuffle=True` i
compara la llista d'errors amb la de `shuffle=False`. Què canvia?

**Exercici 4. Quantes vegades s'encalla?** Entrena un `MLPClassifier` sobre la XOR
amb `random_state` de 0 a 49, per a `hidden_layer_sizes` = `(2,)`, `(3,)` i `(8,)`
(la resta de paràmetres, com a la secció 10). Construeix una taula amb el
percentatge d'entrenaments que resolen la XOR (les quatre prediccions correctes).
Quina conclusió en treus sobre l'amplada de la capa oculta?

**Exercici 5. Una altra activació.** Entrena l'MLP de 8 neurones de la secció 11
amb `activation` = `"logistic"` (la sigmoide), `"tanh"` i `"relu"`. Dibuixa les
tres corbes de loss al mateix gràfic i compara'n l'encert a test i el nombre
d'epochs (`n_iter_`). Quina aprèn més de pressa?

**Exercici 6. Aturada anticipada.** Entrena l'MLP `(16, 16)` de la secció 12 amb
`early_stopping=True`. Ara scikit-learn reserva un 10% de l'entrenament com a
validació (`validation_fraction`) i s'atura quan l'encert de validació deixa de
millorar. Compara `n_iter_`, el temps i l'encert a test amb la versió sense aturada
anticipada. Dibuixa `validation_scores_` al costat de `loss_curve_`.

**Exercici 7. Cercles concèntrics.** Genera dades amb
`sklearn.datasets.make_circles(n_samples=400, noise=0.1, factor=0.5)`. Compara la
regressió logística, el perceptró i un `MLPClassifier` amb diferents mides de capa
oculta. Quin és el nombre mínim de neurones ocultes que resol bé el problema?
Dibuixa les fronteres i intenta explicar per què.

**Exercici 8 (repte). Cercar els millors hiperparàmetres.** Fes servir
`GridSearchCV` per triar, sobre les llunes, la millor combinació de
`hidden_layer_sizes` (`(4,)`, `(8,)`, `(16,)`, `(8, 8)`), `activation` (`"tanh"`,
`"relu"`) i `alpha` (0,0001, 0,01, 1). Quina combinació guanya? Com queda l'encert a
test? Dibuixa la frontera del model guanyador.

## Per al debat de classe

1. **Una regressió logística és una neurona.** Si és així, per què diem que les
   xarxes neuronals són una família d'algorismes diferent i no simplement «moltes
   regressions logístiques»?
2. **El 1958 es va prometre massa** i va venir un hivern. Veieu paral·lelismes
   amb el que es diu avui de la intel·ligència artificial? Què hi ha de diferent?
3. **Precisió o explicació.** Una xarxa encerta el 98% dels casos però ningú no pot
   explicar per què decideix el que decideix. Un decision tree n'encerta el 94%
   i es pot llegir en veu alta. Quin faríeu servir per concedir un crèdit? I per
   detectar pingüins en fotografies aèries? Per què?
4. **Del 1986 al 2012.** La backpropagation ja existia als anys vuitanta, però les
   xarxes neuronals no van dominar fins a l'any 2012. Què creieu que faltava?